# ObsFcstAna Observation-Species Postage Stamps (Animated by Time)

This notebook creates a time animation of `4x4` postage-stamp maps (one frame per ObsFcstAna `.nc4` file).

- Rows: SMAP, SMOS, ASCAT, MODIS+CYGNSS
- Point colors are `obs`
- Each species uses a fixed value range across all files
- Output: GIF (and optional MP4)


In [ ]:
from pathlib import Path
import re
import tempfile
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
try:
    import imageio.v2 as imageio
    HAVE_IMAGEIO = True
except ModuleNotFoundError:
    from PIL import Image as PILImage

    HAVE_IMAGEIO = False

    class _ImageioShim:
        @staticmethod
        def imread(path):
            return np.asarray(PILImage.open(path).convert("RGB"))

        @staticmethod
        def mimsave(out_path, images, duration=0.7, loop=0):
            # Pillow expects per-frame duration in milliseconds.
            frames = [PILImage.fromarray(np.asarray(im).astype(np.uint8)) for im in images]
            if not frames:
                raise RuntimeError("No frames available to write GIF.")
            first, rest = frames[0], frames[1:]
            first.save(
                out_path,
                save_all=True,
                append_images=rest,
                duration=int(max(1, round(duration * 1000))),
                loop=loop,
                optimize=False,
            )

        @staticmethod
        def get_writer(*args, **kwargs):
            raise RuntimeError("MP4 writing requires imageio. Install imageio and imageio-ffmpeg.")

    imageio = _ImageioShim()

import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Input directory
DATA_DIR = Path('/Users/amfox/Desktop/ens_avg/Y2020/M05')
FILE_GLOB = '*.nc4'

# Figure/layout
FIGSIZE = (18, 10)
LAT_MIN = -60.0
POINT_SIZE = 0.8
POINT_ALPHA = 0.55
CMAP_DEFAULT = 'viridis'

# Previous-observation overlay (single gray color)
PREV_WINDOW_HOURS = 22
PREV_POINT_COLOR = '0.65'
PREV_POINT_ALPHA = 0.35
PREV_POINT_SIZE = 0.6

# Species-range controls (fixed across all files)
RANGE_LO_PCT = 2.0
RANGE_HI_PCT = 98.0

# Animation controls
FRAME_DURATION_S = 0.3
GIF_LOOP = 0  # 0 means infinite
WRITE_MP4 = False
FPS_MP4 = 2

OUT_GIF = DATA_DIR / 'obs_species_postage_stamps_animation.gif'
OUT_MP4 = DATA_DIR / 'obs_species_postage_stamps_animation.mp4'

files = sorted(DATA_DIR.glob(FILE_GLOB))
print(f'Found {len(files)} files in {DATA_DIR}')
if not files:
    raise FileNotFoundError(f'No files matched {FILE_GLOB} in {DATA_DIR}')


In [ ]:
def species_names_from_ds(ds):
    if 'obsparam_descr' in ds:
        return [str(v) for v in np.asarray(ds['obsparam_descr'].values).tolist()]

    keys = [k for k in ds.attrs if k.startswith('GEOSldas_observation_species_') and k.endswith('_descr')]
    keys = sorted(keys)
    return [str(ds.attrs[k]) for k in keys]


def clean_obs_fill(obs_arr, obs_var):
    obs = np.asarray(obs_arr, dtype=float)
    fill = obs_var.attrs.get('_FillValue', None)
    missing = obs_var.attrs.get('missing_value', None)
    if fill is not None:
        obs = np.where(obs == float(fill), np.nan, obs)
    if missing is not None:
        obs = np.where(obs == float(missing), np.nan, obs)
    return obs


def parse_stamp(path_obj):
    m = re.search(r'(\d{8}_\d{4}z)', path_obj.name)
    return m.group(1) if m else path_obj.stem


species_order = [
    'SMAP_L1C_Tbh_A', 'SMAP_L1C_Tbh_D', 'SMAP_L1C_Tbv_A', 'SMAP_L1C_Tbv_D',
    'SMOS_fit_Tbh_A', 'SMOS_fit_Tbh_D', 'SMOS_fit_Tbv_A', 'SMOS_fit_Tbv_D',
    'ASCAT_META_SM', 'ASCAT_METB_SM', 'ASCAT_METC_SM', None,
    'MYD10C1', 'MOD10C1', None, 'CYGNSS_SM_6hr',
]

panel_labels = [
    '(a)', '(b)', '(c)', '(d)',
    '(e)', '(f)', '(g)', '(h)',
    '(i)', '(j)', '(k)', '(l)',
    '(m)', '(n)', '(o)', '(p)',
]

row_titles = ['SMAP', 'SMOS', 'ASCAT', 'MODIS + CYGNSS']


In [ ]:
# Pass 1: fixed per-species ranges across all times
species_values_all = {s: [] for s in species_order if s is not None}

for f in files:
    with xr.open_dataset(f) as ds:
        names = species_names_from_ds(ds)
        id_to_name = {i + 1: names[i] for i in range(len(names))}

        sp = np.asarray(ds['species'].values, dtype=np.int32)
        lon = np.asarray(ds['lon'].values, dtype=float)
        lat = np.asarray(ds['lat'].values, dtype=float)
        obs = clean_obs_fill(ds['obs'].values, ds['obs'])

        valid_base = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(obs) & (lat >= LAT_MIN)

        for sid, sname in id_to_name.items():
            if sname not in species_values_all:
                continue
            m = valid_base & (sp == sid)
            if np.any(m):
                species_values_all[sname].append(obs[m])

species_limits = {}
for sname in species_values_all:
    chunks = species_values_all[sname]
    if chunks:
        vals = np.concatenate(chunks)
        lo = float(np.nanpercentile(vals, RANGE_LO_PCT))
        hi = float(np.nanpercentile(vals, RANGE_HI_PCT))
        if not np.isfinite(lo) or not np.isfinite(hi) or (hi <= lo):
            lo = float(np.nanmin(vals))
            hi = float(np.nanmax(vals))
            if not np.isfinite(lo) or not np.isfinite(hi) or (hi <= lo):
                lo, hi = 0.0, 1.0
    else:
        lo, hi = 0.0, 1.0
    species_limits[sname] = (lo, hi)

print('Fixed ranges by species:')
for sname, (lo, hi) in species_limits.items():
    print(f'  {sname:18s} [{lo:.4g}, {hi:.4g}]')


In [ ]:
def parse_stamp_dt(path_obj):
    stamp = parse_stamp(path_obj)
    try:
        return datetime.strptime(stamp, '%Y%m%d_%H%Mz')
    except ValueError:
        return None


def extract_species_payload(nc_path):
    with xr.open_dataset(nc_path) as ds:
        names = species_names_from_ds(ds)
        id_to_name = {i + 1: names[i] for i in range(len(names))}

        sp = np.asarray(ds['species'].values, dtype=np.int32)
        lon = np.asarray(ds['lon'].values, dtype=float)
        lat = np.asarray(ds['lat'].values, dtype=float)
        obs = clean_obs_fill(ds['obs'].values, ds['obs'])

        valid_base = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(obs) & (lat >= LAT_MIN)

        per_species = {}
        for sid, sname in id_to_name.items():
            if sname not in species_limits:
                continue
            m = valid_base & (sp == sid)
            per_species[sname] = {
                'lon': lon[m],
                'lat': lat[m],
                'obs': obs[m],
            }

    return per_species


# Cache all file payloads once so frame rendering can include prior-window overlays.
file_records = []
for f in files:
    file_records.append(
        {
            'path': f,
            'stamp': parse_stamp(f),
            'dt': parse_stamp_dt(f),
            'payload': extract_species_payload(f),
        }
    )

# Keep chronological order when all timestamps are parseable.
if all(rec['dt'] is not None for rec in file_records):
    file_records = sorted(file_records, key=lambda r: r['dt'])

prev_window = timedelta(hours=PREV_WINDOW_HOURS)


def previous_species_xy(frame_idx, sname):
    if sname is None:
        return np.array([], dtype=float), np.array([], dtype=float)

    t_now = file_records[frame_idx]['dt']
    if t_now is None:
        # If timestamp parsing fails, fall back to previous files by index.
        i0 = max(0, frame_idx - int(np.ceil(PREV_WINDOW_HOURS / 3)))
        lon_chunks = [file_records[j]['payload'].get(sname, {}).get('lon', np.array([], dtype=float)) for j in range(i0, frame_idx)]
        lat_chunks = [file_records[j]['payload'].get(sname, {}).get('lat', np.array([], dtype=float)) for j in range(i0, frame_idx)]
    else:
        t_lo = t_now - prev_window
        lon_chunks = []
        lat_chunks = []
        for j, rec in enumerate(file_records):
            if j == frame_idx or rec['dt'] is None:
                continue
            if (rec['dt'] >= t_lo) and (rec['dt'] < t_now):
                data = rec['payload'].get(sname)
                if data is not None:
                    lon_chunks.append(data['lon'])
                    lat_chunks.append(data['lat'])

    if lon_chunks:
        return np.concatenate(lon_chunks), np.concatenate(lat_chunks)
    return np.array([], dtype=float), np.array([], dtype=float)


def render_frame_png(frame_idx, png_path):
    rec = file_records[frame_idx]
    per_species = rec['payload']

    fig, axes = plt.subplots(
        4, 4, figsize=FIGSIZE, subplot_kw={'projection': ccrs.Robinson()}, constrained_layout=True
    )

    for idx, (ax, sname) in enumerate(zip(axes.ravel(), species_order)):
        row = idx // 4

        ax.set_global()
        ax.set_extent([-180, 180, LAT_MIN, 90], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND, facecolor='0.93', zorder=0)
        ax.add_feature(cfeature.OCEAN, facecolor='white', zorder=0)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.30, zorder=2)
        ax.add_feature(cfeature.BORDERS, linewidth=0.20, zorder=2)

        if sname is None:
            ax.set_title('')
        else:
            data = per_species.get(sname, {'lon': np.array([]), 'lat': np.array([]), 'obs': np.array([])})
            x_now = data['lon']
            y_now = data['lat']
            z_now = data['obs']
            n_now = int(z_now.size)

            x_prev, y_prev = previous_species_xy(frame_idx, sname)
            n_prev = int(x_prev.size)

            if n_prev > 0:
                ax.scatter(
                    x_prev,
                    y_prev,
                    s=PREV_POINT_SIZE,
                    color=PREV_POINT_COLOR,
                    alpha=PREV_POINT_ALPHA,
                    linewidths=0,
                    transform=ccrs.PlateCarree(),
                    zorder=1,
                    rasterized=True,
                )

            vmin, vmax = species_limits[sname]
            if n_now > 0:
                ax.scatter(
                    x_now,
                    y_now,
                    c=z_now,
                    s=POINT_SIZE,
                    alpha=POINT_ALPHA,
                    cmap=CMAP_DEFAULT,
                    vmin=vmin,
                    vmax=vmax,
                    linewidths=0,
                    transform=ccrs.PlateCarree(),
                    zorder=2,
                    rasterized=True,
                )

            ax.set_title(f"{panel_labels[idx]} {sname}", fontsize=8.2, loc="left")
            ax.text(
                0.985,
                0.02,
                f"Count: {n_now:>8,d}",
                transform=ax.transAxes,
                ha="right",
                va="bottom",
                fontsize=7.2,
                fontfamily="monospace",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.80, pad=0.9),
                zorder=3,
            )

        if idx % 4 == 0:
            ax.text(
                -0.16,
                0.5,
                row_titles[row],
                transform=ax.transAxes,
                rotation=90,
                va='center',
                ha='center',
                fontsize=10,
                fontweight='bold',
            )

    fig.suptitle(f"ObsFcstAna Observation Species | {rec['stamp']}", fontsize=14)
    fig.savefig(png_path, dpi=180, bbox_inches='tight')
    plt.close(fig)


from PIL import Image as PILImage

with tempfile.TemporaryDirectory(prefix='postage_frames_') as tmpd:
    tmpd = Path(tmpd)
    frame_paths = []

    for i, _ in enumerate(file_records, start=1):
        frame_png = tmpd / f'frame_{i:04d}.png'
        render_frame_png(i - 1, frame_png)
        frame_paths.append(frame_png)
        print(f'[{i:03d}/{len(file_records):03d}] rendered {file_records[i - 1]["path"].name}')

    # Write GIF with Pillow so frame delay is explicit and consistent across viewers.
    gif_frames = [PILImage.open(p).convert('P', palette=PILImage.ADAPTIVE) for p in frame_paths]
    if not gif_frames:
        raise RuntimeError('No frames available to write GIF.')
    gif_duration_ms = int(max(1, round(FRAME_DURATION_S * 1000)))
    gif_frames[0].save(
        OUT_GIF,
        save_all=True,
        append_images=gif_frames[1:],
        duration=gif_duration_ms,
        loop=GIF_LOOP,
        optimize=False,
    )
    print(f'Wrote GIF: {OUT_GIF}')

    if WRITE_MP4 and (not HAVE_IMAGEIO):
        print('WRITE_MP4=True but imageio is unavailable; skipping MP4.')

    if WRITE_MP4 and HAVE_IMAGEIO:
        with imageio.get_writer(OUT_MP4, fps=FPS_MP4, codec='libx264') as writer:
            for p in frame_paths:
                writer.append_data(imageio.imread(p))
        print(f'Wrote MP4: {OUT_MP4}')


In [ ]:
from IPython.display import Image, Video, display
from PIL import Image as PILImage

if OUT_GIF.exists():
    print(f"Preview GIF: {OUT_GIF}")

    with PILImage.open(OUT_GIF) as _gif:
        nframes = getattr(_gif, "n_frames", 1)
        frame_delays = []
        for i in range(nframes):
            _gif.seek(i)
            frame_delays.append(_gif.info.get("duration", None))

    unique_delays = sorted({d for d in frame_delays if d is not None})
    if unique_delays:
        if len(unique_delays) == 1:
            d = unique_delays[0]
            print(f"GIF metadata: frames={nframes}, delay={d} ms/frame (~{d/1000:.2f} s)")
        else:
            print(f"GIF metadata: frames={nframes}, delays(ms)={unique_delays}")
    else:
        print(f"GIF metadata: frames={nframes}, no per-frame delay metadata found")

    # Display from raw bytes to avoid stale file-name caching in notebook frontends.
    display(Image(data=OUT_GIF.read_bytes(), format="gif"))
else:
    print(f"GIF not found yet: {OUT_GIF}")

if WRITE_MP4:
    if OUT_MP4.exists():
        print(f"Preview MP4: {OUT_MP4}")
        display(Video(filename=str(OUT_MP4), embed=True))
    else:
        print(f"MP4 not found yet: {OUT_MP4}")
